## Quantum Support Vector Machine (QSVM)

In [2]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 145.9 kB/s  0:01:25m0:00:0200:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [7]:
!pip install qiskit-machine-learning

  Using cached qiskit_machine_learning-0.8.4-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.0.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached qiskit_machine_learning-0.8.4-py3-none-any.whl (231 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 30.4 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4━━━━━ 0/2 [numpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [qiskit-machine-learning]-machine-learning]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
qctrl-visualizer 8.0.2 requires numpy<2.0.0,>=1.26.0, but you have numpy 2.0.2 which is incompatible.


In [8]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

from qiskit_aer import AerSimulator
from qiskit.primitives import Sampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.circuit.library import ZZFeatureMap

## Dataset (Binary Classification)

In [9]:
# Load dataset (binary)
X, y = datasets.load_breast_cancer(return_X_y=True)

# Reduce dimension for quantum circuit (important)
X = X[:, :2]

# Normalize
X = (X - X.min()) / (X.max() - X.min())

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Quantum Feature Map

In [10]:
feature_map = ZZFeatureMap(feature_dimension=2, reps=2)

## Quantum Kernel

In [14]:
from qiskit.primitives import Sampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute

# Step 1: Create sampler
sampler = Sampler()

# Step 2: Create fidelity object
fidelity = ComputeUncompute(sampler=sampler)

# Step 3: Create quantum kernel
quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    fidelity=fidelity
)

/tmp/ipykernel_1156/1101502941.py:9: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  fidelity = ComputeUncompute(sampler=sampler)


### Train QSVM

In [15]:
# Use classical SVM with quantum kernel
qsvm = SVC(kernel=quantum_kernel.evaluate)

qsvm.fit(X_train, y_train)

accuracy_qsvm = qsvm.score(X_test, y_test)

print("QSVM Accuracy:", accuracy_qsvm)

QSVM Accuracy: 0.8859649122807017
